In [1]:
import torch
import torch.nn as nn

c:\Users\Manoj\Python ML\Lib\site-packages\torch\cuda\__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
class RoPEv2(nn.Module):
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        assert dim % 2 == 0, "Dimension must be even"

        self.dim = dim
        self.base = base
        self.max_seq_len = max_seq_len

        # inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

        # positions
        positions = torch.arange(max_seq_len).float()

        # angles: (max_seq_len, dim/2)
        angles = torch.einsum("i,j->ij", positions, inv_freq)

        # precompute sin/cos
        sin = torch.sin(angles)[None, None, :, :]  # (1, 1, seq_len, dim/2)
        cos = torch.cos(angles)[None, None, :, :]
        self.register_buffer("sin", sin)
        self.register_buffer("cos", cos)

    def forward(self, x):
        """
        (batch, num_heads, seq_len, dim)
        """
        seq_len = x.shape[2]

        # extend cache if needed
        if seq_len > self.max_seq_len:
            new_len = seq_len

            positions = torch.arange(self.max_seq_len, new_len).float().to(x.device)
            angles = torch.einsum("i,j->ij", positions, self.inv_freq)

            sin_new = torch.sin(angles)[None, None, :, :]
            cos_new = torch.cos(angles)[None, None, :, :]

            self.sin = torch.cat([self.sin.to(x.device), sin_new], dim=2)
            self.cos = torch.cat([self.cos.to(x.device), cos_new], dim=2)

            self.max_seq_len = new_len

        # slice
        sin = self.sin[:, :, :seq_len, :]  # (1, 1, seq_len, dim/2)
        cos = self.cos[:, :, :seq_len, :]

        # split
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        # rotation
        x_rotated = torch.stack([
            x1 * cos - x2 * sin,
            x1 * sin + x2 * cos
        ], dim=-1)

        return x_rotated.flatten(-2)

In [3]:
class DeepSeekAttention(nn.Module):
  """
   The full, state-of-the-art attention mechanism from DeepSeek, combining
    Multi-Head Latent Attention (MLA) with Decoupled Rotary Positional
    Encoding (RoPE).
  """
  def __init__(self, d_model, num_heads, d_latent, d_rope, dropout=0.0, max_seq_len=2048):
    super().__init__()
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads
    self.d_latent = d_latent
    self.d_rope = d_rope

    # --- A: Content Path (Pure MLA) ---
    self.W_q = nn.Linear(d_model, d_model)
    self.W_dkv = nn.Linear(d_model, d_latent)
    self.W_uk = nn.Linear(d_latent, d_model)
    self.W_uv = nn.Linear(d_latent, d_model)

    # --- B: Position Path (RoPE Applied) ---
    self.W_k_pos = nn.Linear(d_model, d_rope * num_heads)
    self.W_q_pos = nn.Linear(d_model, d_rope * num_heads)

    # RoPE module to apply the rotations
    self.rope = RoPEv2(d_rope)

    # --- C: Final Output Projection ---
    self.W_o = nn.Linear(d_model, d_model)

    self.dropout = nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(1, 1,  max_seq_len,  max_seq_len), diagonal=1).bool())

  def forward(self, x):
    batch_size, num_tokens, _ = x.shape

    # --- A: Content Path Calculation ---
    # This path is cache-friendly and position-agnostic.
    q_c = self.W_q(x).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
    c_kv = self.W_dkv(x) # This is what gets cached for the content path.
    k_c = self.W_uk(c_kv).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
    v_c = self.W_uv(c_kv).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

    # --- B: Position Path Calculation ---
    # This path handles the positional information.
    q_r_unrotated = self.W_q_pos(x).view(batch_size, num_tokens, self.num_heads, self.d_rope).transpose(1, 2)
    k_r_unrotated = self.W_k_pos(x).view(batch_size, num_tokens, self.num_heads, self.d_rope).transpose(1, 2)

    # Apply Rope to positional keys and query vectors
    q_r = self.rope(q_r_unrotated)
    k_r = self.rope(k_r_unrotated)   # This is what gets cached for the position path.

    # --- C: Combining Paths for Final Attention Score ---
    # The final score is the sum of content and position scores.
    content_scores = (q_c @ k_c.transpose(2, 3)) / (self.head_dim ** 0.5)
    position_scores = (q_r @ k_r.transpose(2, 3)) / (self.d_rope ** 0.5)

    attn_scores = content_scores + position_scores

    # --- D: Final Steps (Masking, Softmax, Output) ---
    attn_scores = attn_scores.masked_fill(
        self.mask[:, :, :seq_len, :seq_len], float('-inf'))

    attn_weights = torch.softmax(attn_scores, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # The final context vector is computed using only the content value matrix (v_c) it didn't need rope
    context_vector = (attn_weights @ v_c).transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_model)

    output = self.W_o(context_vector)
    return output


In [4]:
# --- Usage Example ---
d_model = 512
num_heads = 8
d_latent = 128
d_rope = 64 # Dimension for RoPE, typically d_head or smaller
batch_size = 4
seq_len = 64

# Instantiate the full attention layer
deepseek_attn_layer = DeepSeekAttention(d_model, num_heads, d_latent, d_rope)

# Create a dummy input tensor
dummy_input = torch.randn(batch_size, seq_len, d_model)  #[4, 64, 512]

# Pass the input through the layer
output = deepseek_attn_layer(dummy_input)

print("DeepSeekAttention Layer successful!")
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

DeepSeekAttention Layer successful!
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])
